In [ ]:
### Logistic Regression
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

FRIDAY_PATH = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
WEDNESDAY_PATH = "Wednesday-workingHours.pcap_ISCX.csv"

df_fri = pd.read_csv(FRIDAY_PATH)
df_wed = pd.read_csv(WEDNESDAY_PATH)
data = pd.concat([df_fri, df_wed], ignore_index=True)
print(f"Combined shape: {data.shape}")

data.columns = data.columns.str.strip()
if "Label" not in data.columns:
    raise ValueError("Expected 'Label' column not found.")

data["Label"] = data["Label"].astype(str).str.strip()
data["Label"] = data["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

data = data.apply(pd.to_numeric, errors="coerce")
data.replace([np.inf, -np.inf], np.nan, inplace=True)

before = data.shape[0]
data.dropna(inplace=True)
after = data.shape[0]
print(f"Dropped {before - after} rows due to NaN/inf values. Remaining rows: {after}")

zero_var_cols = [c for c in data.columns if c != "Label" and data[c].nunique() <= 1]
if zero_var_cols:
    print("Dropping zero-variance columns:", zero_var_cols)
    data.drop(columns=zero_var_cols, inplace=True)

feature_names = [c for c in data.columns if c != "Label"]
X = data[feature_names].values
y = data["Label"].values

print(f"Final feature count: {len(feature_names)}")
print(f"Final dataset size  : {data.shape[0]} rows")

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
print(f"Shapes -> TrainVal: {X_trainval.shape}, Test: {X_test.shape}")

scaler = MinMaxScaler()
X_trainval = scaler.fit_transform(X_trainval)
X_test = scaler.transform(X_test)

logreg_base = LogisticRegression(
    penalty="l2",
    solver="saga",       
    max_iter=2000,
    random_state=SEED,
    n_jobs=-1
)

param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0, 100.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

grid = GridSearchCV(
    estimator=logreg_base,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True,
    return_train_score=False
)

print("\n CV hyperparameter optimization")
grid.fit(X_trainval, y_trainval)

print("Best CV accuracy:", f"{grid.best_score_:.6f}")
print("Best params:", grid.best_params_)

logreg_clf = grid.best_estimator_

y_pred = logreg_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
far = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"FAR      : {far:.6f}")